In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
# import torch
# import torch.nn.functional as F
# from torch_geometric.nn import GraphConv
# from torch.nn import Linear
# from torch_geometric.nn import global_mean_pool
import urllib.request

Create ppi human graph

In [2]:
df = pd.read_csv('bio-pathways-network.csv')

edges = list(zip(df['Gene ID 1'], df['Gene ID 2']))

G_human = nx.Graph()
G_human.add_edges_from(edges)

print("Human nodes:", len(G_human.nodes()), "edges:", len(G_human.edges()))

Human nodes: 21557 edges: 342353


Retrieve and convert yeast file to graph

In [3]:
urllib.request.urlretrieve(
    'http://snap.stanford.edu/deepnetbio-ismb/ipynb/yeast.edgelist',
    'yeast.edgelist'
)
yeast_file = "yeast.edgelist"
G_yeast = nx.read_edgelist(yeast_file)

print("Yeast nodes:", len(G_yeast.nodes()), "edges:", len(G_yeast.edges()))

Yeast nodes: 6526 edges: 532180


Graph Embedding, Labelling and feature extraction

In [4]:
def create_labels(G, threshold=5):
    labels = {}
    for node, deg in dict(G.degree()).items():
        labels[node] = 1 if deg >= threshold else 0
    return labels

labels_yeast = create_labels(G_yeast, threshold=5)
labels_human = create_labels(G_human, threshold=10)

#print graph label
print("Yeast labels:", list(labels_yeast.items())[:10])
print("Human labels:", list(labels_human.items())[:10])

Yeast labels: [('YLR418C', 1), ('YOL145C', 1), ('YOR123C', 1), ('YBR279W', 1), ('YML069W', 1), ('YGL244W', 1), ('YGL207W', 1), ('YER164W', 1), ('YIL035C', 1), ('YOR061W', 1)]
Human labels: [(1394, 1), (2778, 1), (6331, 1), (17999, 1), (122704, 1), (54460, 1), (2597, 1), (2911, 1), (4790, 1), (79155, 1)]


In [ ]:
from node2vec import Node2Vec
yeast_emb = Node2Vec(G_yeast, dimensions=4, walk_length=30, num_walks=20, workers=8)

c:\Users\tiffa\anaconda3\envs\ppi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Computing transition probabilities: 100%|██████████| 6526/6526 [55:48<00:00,  1.95it/s]   
c:\Users\tiffa\anaconda3\envs\ppi\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
